# SOXX 披露持仓分散度的原 Stage1 匹配信息检验

冻结机制、来源、对照、标签成熟和预算见 `plan.json`。

In [1]:
import os,sys,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib,sklearn
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings
assert 'qlib_venv' in sys.executable,sys.executable
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd();card=root/'research/stage1_soxx_reported_diversity_information_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
seal=json.loads((card/'implementation_seal.json').read_text())
current_code=''.join(json.loads((card/'stage1_soxx_reported_diversity_information_20261023.ipynb').read_text())['cells'][1]['source'])
code_sha=hashlib.sha256(current_code.encode()).hexdigest()
assert seal['plan_sha256']==sha(card/'plan.json') and seal['code_cell_source_sha256']==code_sha
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and sklearn.__version__==plan['model']['runtime_scikit_learn']
assert not (card/'run_state.json').exists() and not (card/'preflight_result.json').exists()
parent=plan['parent'];parent_card=root/parent['source_card'];parent_art=parent_card/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'
for name,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:
 assert sha(parent_art/name)==parent[key],('parent artifact drift',name)
for part in ('train_source','held_source'):
 spec=parent[part];c=root/spec['card'];summary=json.loads((c/'summary.json').read_text())
 assert sha(c/'summary.json')==spec['summary_sha256'] and sha(c/'source_audit.json')==spec['source_audit_sha256']
 assert summary.get('source_NAS_evidence',summary.get('nas_evidence'))['sha256']==spec['NAS_archive_sha256'] and summary['nas_readback']['verified']
 art=c/'qlib_recorder'/spec['experiment_id']/spec['recorder_id']/'artifacts/analysis/mapped_signals'
 assert sha(art)==spec['mapped_artifact_sha256']
ctx=plan['source_index_change_context'];assert sha(root/ctx['path'])==ctx['sha256']
qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
cal=pd.DatetimeIndex(D.calendar(start_time=plan['cohorts']['train_signal'][0],end_time=plan['cohorts']['held_reused_diagnostic'][1],freq='day'))
train_calendar=cal[(cal>=pd.Timestamp(plan['cohorts']['train_signal'][0]))&(cal<=pd.Timestamp(plan['cohorts']['train_signal'][1]))]
held_calendar=cal[(cal>=pd.Timestamp(plan['cohorts']['held_reused_diagnostic'][0]))&(cal<=pd.Timestamp(plan['cohorts']['held_reused_diagnostic'][1]))]
assert len(train_calendar)==plan['cohorts']['expected_train_source_rows']==3020
assert len(held_calendar)==plan['cohorts']['expected_held_source_rows']==944
with R.uri_context(parent['train_source']['recorder_uri']):
 spec=parent['train_source'];train_map=R.get_recorder(recorder_id=spec['recorder_id'],experiment_id=spec['experiment_id']).load_object('analysis/mapped_signals')
with R.uri_context(parent['held_source']['recorder_uri']):
 spec=parent['held_source'];held_map=R.get_recorder(recorder_id=spec['recorder_id'],experiment_id=spec['experiment_id']).load_object('analysis/mapped_signals')
with R.uri_context(str(parent_card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id'])
 original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features')
 frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
assert len(train_map)==3020 and len(held_map)==944
assert pd.DatetimeIndex(train_map.signal_date).equals(train_calendar) and pd.DatetimeIndex(held_map.signal_date).equals(held_calendar)
assert train_map.qualified_source.all() and not train_map.amendment_uncertain.any() and held_map.source_known.all()
for mapped in (train_map,held_map):
 assert mapped.signal_date.is_unique and (mapped.effective_signal_date<=mapped.signal_date).all()
 assert (mapped.signal_date>mapped.filed_date).all() and mapped.source_age_days.ge(1).all()
 assert np.isfinite(mapped[['top5','HHI','source_age_days']].to_numpy(float)).all()
 assert mapped.top5.between(0,1).all()
base_names=list(original.base_features);assert len(base_names)==39 and len(set(base_names))==39
assert set(base_names).issubset(base.columns) and frame.index.equals(base.index) and frame.index.equals(timing.index)
assert set(cal).issubset(set(frame.index))
source=pd.concat([train_map,held_map],ignore_index=True).set_index('signal_date').reindex(frame.index)
name=plan['single_mechanism']['feature'];candidate=(1.0-source.top5.astype(float)).rename(name)
controls=pd.DataFrame(index=frame.index)
controls['source_age_days']=source.source_age_days.astype(float)
controls['NPORT_form_indicator']=source.form.fillna('').astype(str).str.startswith('NPORT').astype(float)
controls['post_announced_expected_2021_index_change']=(frame.index>=pd.Timestamp(ctx['announced_expected_date'])).astype(float)
forward=frame['soxx_forward_open_return'].astype(float)
parts=[forward.shift(-i) for i in range(5)]
valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC')
last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_mask=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<cutoff)&label.notna()
held_mask=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<last)&label.notna()
assert train_mask.sum()==plan['cohorts']['expected_mature_train_rows']==3014
assert held_mask.sum()==plan['cohorts']['expected_mature_held_rows']==940
eligible=train_mask|held_mask
assert candidate.loc[eligible].notna().all() and candidate.loc[eligible].between(0,1).all()
assert controls.loc[eligible].notna().all().all() and source.loc[eligible,'source_age_days'].ge(1).all()
assert set(base_names).isdisjoint({name})
augmented=base[base_names].copy()
for col in controls.columns:augmented[col]=controls[col]
augmented[name]=candidate
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_mask,'held_eligible':held_mask},index=frame.index)
static=pd.concat({'feature':augmented,'label':label.to_frame(),'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])})
tr_all=dataset.prepare('train',col_set=['feature','label','meta']);he_all=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(tr_all)==3020 and len(he_all)==944
tr=tr_all.loc[tr_all[('meta','train_eligible')].astype(bool)];he=he_all.loc[he_all[('meta','held_eligible')].astype(bool)]
assert len(tr)==3014 and len(he)==940
preflight={'status':'matched_original_Stage1_39_plus_shared_controls_Qlib_source_and_H5_maturity_passed',
 'plan_sha256':sha(card/'plan.json'),'implementation_code_sha256':code_sha,'original_primary_features':39,'common_controls':list(controls.columns),
 'train_source_rows':len(train_map),'held_source_rows':len(held_map),'train_mature_rows':len(tr),'held_mature_rows':len(he),
 'train_candidate_missing':int(candidate.loc[train_mask].isna().sum()),'held_candidate_missing':int(candidate.loc[held_mask].isna().sum()),
 'train_last_label_available_at':str(mature.loc[train_mask].max()),'held_last_label_available_at':str(mature.loc[held_mask].max()),
 'train_source_age_max_days':int(source.loc[train_mask,'source_age_days'].max()),'held_source_age_max_days':int(source.loc[held_mask,'source_age_days'].max()),
 'source_Recorder_object_loads':6,'DatasetH_builds':1,'original_Stage1_H5_label_source_artifact_loads':1,'model_fits_before_preflight':0,'native_accounts':0,
 'historical_market_independence':False,'historical_exact_intraday_source_clock_qualified':False}
(card/'preflight_result.json').write_text(json.dumps(preflight,indent=2)+'\n')
state_path=card/'run_state.json';state={'plan_sha256':sha(card/'plan.json'),'implementation_code_sha256':code_sha,'fit_attempts':[],'fits_done':[],'bootstrap_attempted':False,'new_native_accounts':0}
state_path.write_text(json.dumps(state,indent=2)+'\n')
yname=label.name;ytr=tr[('label',yname)].to_numpy(float);yhe=he[('label',yname)].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
controls_names=list(controls.columns)
inputs={'baseline':(tr['feature'][base_names+controls_names].to_numpy(float),he['feature'][base_names+controls_names].to_numpy(float)),
 'candidate':(tr['feature'][base_names+controls_names+[name]].to_numpy(float),he['feature'][base_names+controls_names+[name]].to_numpy(float))}
assert np.isfinite(inputs['candidate'][0][:,-1]).all() and np.isfinite(inputs['candidate'][1][:,-1]).all()
models={};predictions={};tracking=card/'qlib_recorder';dataset.config(dump_all=True,recursive=True)
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 recorder=R.get_recorder();state['recorder_id']=recorder.id;state['experiment_id']=str(recorder.experiment_id)
 state_path.write_text(json.dumps(state,indent=2)+'\n')
 recorder.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset,
  train_source_receipt=parent['train_source'],held_source_receipt=parent['held_source'])
 recorder.log_metrics(train_mature_rows=len(tr),held_mature_rows=len(he),model_fits=0,native_accounts=0)
 for family in ('baseline','candidate'):
  assert family not in state['fit_attempts'] and len(state['fit_attempts'])<plan['budget']['model_fit_attempts_max']
  state['fit_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
  model.fit(inputs[family][0],ytr)
  prediction=model.predict(inputs[family][1]);assert np.isfinite(prediction).all()
  models[family]=model;predictions[family]=prediction
  recorder.save_objects(artifact_path='models',**{family:model})
  recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':prediction,'target':yhe})})
  state['fits_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  recorder.log_metrics(completed_fits=len(state['fits_done']))
 dates=pd.DatetimeIndex(he.index.get_level_values('datetime'))
 daily=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
 daily['baseline_squared_error']=(daily.target-daily.baseline_prediction)**2
 daily['candidate_squared_error']=(daily.target-daily.candidate_prediction)**2
 daily['paired_mse_improvement']=daily.baseline_squared_error-daily.candidate_squared_error
 assert daily.signal_date.is_monotonic_increasing and len(daily)==940
 boundary=pd.Timestamp('2025-01-01');halves=[daily.loc[daily.signal_date.lt(boundary)],daily.loc[daily.signal_date.ge(boundary)]]
 assert min(map(len,halves))>300
 bootstrap=plan['information_gate']['paired_stationary_bootstrap']
 state['bootstrap_attempted']=True;state_path.write_text(json.dumps(state,indent=2)+'\n')
 interval,_,replicates=full_calendar_bootstrap(daily.paired_mse_improvement.to_numpy(float),
  mean_block=bootstrap['mean_block_sessions'],resamples=bootstrap['draws'],seed=bootstrap['seed'],alpha=bootstrap['one_sided_alpha'])
 coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
 metrics={'baseline_held_mse':float(daily.baseline_squared_error.mean()),'candidate_held_mse':float(daily.candidate_squared_error.mean()),
  'held_paired_mse_improvement':float(daily.paired_mse_improvement.mean()),
  'held_half_paired_improvements':[float(x.paired_mse_improvement.mean()) for x in halves],
  'candidate_standardized_coordinate_coefficient':coefficient,
  'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
 gates={'coefficient_positive':coefficient>0,'full_held_mean_positive':metrics['held_paired_mse_improvement']>0,
  'both_halves_positive':all(x>0 for x in metrics['held_half_paired_improvements']),
  'bootstrap_lower_positive':metrics['bootstrap_lower']>0}
 passed=bool(all(gates.values()))
 daily['quarter']=daily.signal_date.dt.to_period('Q').astype(str)
 quarters=[{'quarter':q,'sessions':len(group),'paired_mean_mse_improvement':float(group.paired_mse_improvement.mean())}
  for q,group in daily.groupby('quarter',sort=True)]
 recorder.save_objects(artifact_path='analysis/daily',held=daily)
 recorder.save_objects(artifact_path='analysis',metrics=metrics,gates=gates,quarters=quarters,
  bootstrap_replicates=replicates,information_gate_pass=passed)
 recorder.log_metrics(held_paired_mse_improvement=metrics['held_paired_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
summary={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),
 'decision':'conditional_development_reported_diversity_information_pass_account_pending' if passed else 'not_promoted_fixed_reported_diversity_information_failed',
 'objective_achieved':False,'implementation_code_sha256':code_sha,'information_gate_pass':passed,'training_mature_rows':len(tr),'held_mature_rows':len(he),
 'metrics':metrics,'gates':gates,'quarterly_information_attribution':quarters,
 'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'reused_market_history':True,
 'historical_exact_intraday_source_clock_qualified':False,'independent_forward_qualified_days':0,
 'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),
 'recorder_local_dir':str(tracking/state['experiment_id']/state['recorder_id']),
 'artifact_policy':plan['artifact_policy'],
 'limitations':['Original SOXX disclosed equity basket is a delayed ETF proxy, not daily index weights or realized breadth',
  'Conservative SEC filing-date next-session timing; exact historical intraday first-public clock is not certified',
  'Held 2023-2026 market history is repeatedly reused development feedback, not independent validation',
  'Information MSE does not establish a funded fee-net SOXX account result']}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
(card/'result_review.json').write_text(json.dumps({'status':summary['decision'],'plan_sha256':summary['plan_sha256'],'recorder_id':summary['recorder_id'],'metrics':metrics,'gates':gates,'new_model_fits':2,'new_native_accounts':0},ensure_ascii=False,indent=2)+'\n')
pd.DataFrame(quarters).to_csv(card/'quarterly_information_attribution.csv',index=False)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
 stored_metrics=checked.load_object('analysis/metrics');stored_gates=checked.load_object('analysis/gates')
 assert stored_metrics==metrics and stored_gates==gates
 assert len(checked.load_object('predictions/baseline'))==len(checked.load_object('predictions/candidate'))==940
settings=_settings(root);binding=card/'nas_binding.json'
binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='reported-diversity-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:
  run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as saved:names=[m.name for m in saved.getmembers() if m.isfile()]
 assert any(x.endswith('/artifacts/analysis/metrics') for x in names)
 assert any(x.endswith('/artifacts/predictions/candidate') for x in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
state['completed']=True;state_path.write_text(json.dumps(state,indent=2)+'\n')
print({'decision':summary['decision'],'gate_pass':passed,'delta_mse':metrics['held_paired_mse_improvement'],'bootstrap_lower':metrics['bootstrap_lower'],'fits':2,'accounts':0})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[69578:MainThread](2026-10-23 03:25:39,025) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[69578:MainThread](2026-10-23 03:25:39,027) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[69578:MainThread](2026-10-23 03:25:39,027) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[69578:MainThread](2026-10-23 03:25:39,663) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


...

[69578:MainThread](2026-10-23 03:25:39,670) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_soxx_reported_diversity_information_20261023.


[69578:MainThread](2026-10-23 03:25:39,671) INFO - qlib.workflow - [exp.py:258] - Experiment 909397922584822167 starts running ...


[69578:MainThread](2026-10-23 03:25:39,741) INFO - qlib.workflow - [recorder.py:345] - Recorder 9bf200f52a964a0d8f8c307b946d54ca starts running under Experiment 909397922584822167 ...


[69578:MainThread](2026-10-23 03:25:42,578) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'decision': 'not_promoted_fixed_reported_diversity_information_failed', 'gate_pass': False, 'delta_mse': -9.35003963098238e-05, 'bootstrap_lower': -0.0003287968722680612, 'fits': 2, 'accounts': 0}


## 中文结论

固定信息筛选失败：940 行候选 MSE 0.002814042 高于基线 0.002720541，两个半区、系数方向及校正区块下界均失败。2 拟合、1 自举、0 账户；Recorder 9bf200f52a964a0d8f8c307b946d54ca、NAS SHA a79ebc7404798480b3d75f5af3712579ca03796a6cfe1b480243974d2bb155d6。不晋升，不重调本版。
